# Section 122 Exemption & SCOTUS Analysis

Analyzes the coverage and impact of the Section 122 temporary import surcharge,
including product exemptions, country-level effects, and the pre/post SCOTUS comparison.

**Data pipeline**: Reads `country-by-time.csv` (produced by the main tariff notebook)
for the tariff panel. Loads import data from cached parquet files and defines
the mask functions needed for exemption classification.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

## Data Pipeline

In [ ]:
WEIGHT_YEAR = 2024

### Country list

In [ ]:
country_list = pd.read_csv('./data/top_50_non_eu.csv', dtype={'CTY_CODE': str})

# Add EU as a bloc
eu_row = pd.DataFrame([{"CTY_NAME": "EUROPEAN UNION", "CTY_CODE": "0003"}])
country_list = pd.concat([country_list, eu_row], ignore_index=True)

### Product-level tariff lists (for masks)

In [ ]:
steel_list = pd.read_csv(
    './tariff-lists/steel-tariffs.csv', dtype={'HTSUS': str})

alu_list = pd.read_csv(
    './tariff-lists/alu-tariffs.csv', dtype={'HTSUS': str})
alu_8_list = alu_list[alu_list['HTSUS'].str.len() == 8]

auto_list = pd.read_csv(
    './tariff-lists/auto-tariffs.csv', dtype={'HTSUS': str})
auto_6_list = auto_list[auto_list['HTSUS'].str.len() == 6]
auto_8_list = auto_list[auto_list['HTSUS'].str.len() == 8]

copper_list = pd.read_csv(
    './tariff-lists/copper-list.csv', dtype={'HTSUS': str})

s122_exempt_list = pd.read_csv(
    './tariff-lists/section122-product-exemptions.csv', dtype={'hts_code': str})
s122_exempt_8 = s122_exempt_list[s122_exempt_list['digits'] == 8]['hts_code'].tolist()
s122_exempt_10 = s122_exempt_list[s122_exempt_list['digits'] == 10]['hts_code'].tolist()

# April 2026 metals annex lists (needed for is_232 classification)
metals_ia_list = pd.read_csv('./tariff-lists/metals-annex-IA.csv', dtype={'HTSUS': str})
metals_ia_4  = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 4]['HTSUS'].tolist()
metals_ia_6  = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 6]['HTSUS'].tolist()
metals_ia_8  = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 8]['HTSUS'].tolist()
metals_ia_10 = metals_ia_list[metals_ia_list['HTSUS'].str.len() == 10]['HTSUS'].tolist()

metals_ib_list = pd.read_csv('./tariff-lists/metals-annex-IB.csv', dtype={'HTSUS': str})
metals_ib_8  = metals_ib_list[metals_ib_list['HTSUS'].str.len() == 8]['HTSUS'].tolist()
metals_ib_10 = metals_ib_list[metals_ib_list['HTSUS'].str.len() == 10]['HTSUS'].tolist()

metals_iii_list = pd.read_csv('./tariff-lists/metals-annex-III-temporary.csv', dtype={'HTSUS': str})
metals_iii_8  = metals_iii_list[metals_iii_list['HTSUS'].str.len() == 8]['HTSUS'].tolist()
metals_iii_10 = metals_iii_list[metals_iii_list['HTSUS'].str.len() == 10]['HTSUS'].tolist()

### Import data loader

In [ ]:
def load_imports(cnty_code, year=2024, data_dir="./data"):
    infile = f"{data_dir}/{year}-hs10-imports{cnty_code}.parquet"
    return pq.read_table(infile).to_pandas()

### Product-group masks

In [ ]:
def _mask_steel(df):
    return df["HS8"].isin(steel_list["HTSUS"].tolist())

def _mask_alu(df):
    return (df["HS10"].isin(alu_list["HTSUS"].tolist())
            | df["HS8"].isin(alu_8_list["HTSUS"].tolist()))

def _mask_auto(df):
    return (df["HS8"].isin(auto_8_list["HTSUS"].tolist())
            | df["HS10"].isin(auto_list["HTSUS"].tolist())
            | df["HS6"].isin(auto_6_list["HTSUS"].tolist()))

def _mask_copper(df):
    return df["HS8"].isin(copper_list["HTSUS"].tolist())

def _mask_s122_exempt(df):
    return df["HS8"].isin(s122_exempt_8) | df["HS10"].isin(s122_exempt_10)

def _mask_annex_IA(df):
    return (df["HS10"].isin(metals_ia_10)
            | df["HS8"].isin(metals_ia_8)
            | df["HS6"].isin(metals_ia_6)
            | df["HS8"].str[:4].isin(metals_ia_4))

def _mask_annex_IB(df):
    return (df["HS10"].isin(metals_ib_10)
            | df["HS8"].isin(metals_ib_8))

def _mask_annex_III(df):
    return (df["HS10"].isin(metals_iii_10)
            | df["HS8"].isin(metals_iii_8))

### Load tariff panel

Reads `country-by-time.csv` produced by the main tariff notebook.

In [ ]:
tariffs_by_date = pd.read_csv('country-by-time.csv', parse_dates=['date'])
tariffs_by_date.set_index('date', inplace=True)
tariffs_by_date.head()

## Section 122 Exemption Analysis

Compute the share of U.S. imports (by value) that are exempt from the
Section 122 temporary import surcharge under each exemption category.

In [ ]:
# ── Section 122 Exemption Coverage ─────────────────────────────────────
# For each country, load 2024 imports and classify every HS10 line into:
#   (a) Section 232 products (steel, alu, autos, copper) — carved out
#   (b) Annex I (aa)(ii) product exemptions — exempt from surcharge
#   (c) USMCA (Canada/Mexico) — fully exempt
#   (d) Remainder — subject to 10% surcharge

rows_ex = []

for _, row in country_list.iterrows():
    cty = row['CTY_CODE']
    df = load_imports(cty, year=WEIGHT_YEAR)

    total = df['imports'].sum()
    if total == 0:
        continue

    is_232 = _mask_steel(df) | _mask_alu(df) | _mask_auto(df) | _mask_copper(df)
    is_s122_exempt = _mask_s122_exempt(df)
    is_usmca = cty in ('1220', '2010')

    imp_232 = df.loc[is_232, 'imports'].sum()
    imp_annex = df.loc[is_s122_exempt & ~is_232, 'imports'].sum()

    if is_usmca:
        imp_usmca = total
        imp_subject = 0.0
    else:
        imp_usmca = 0.0
        imp_subject = total - imp_232 - imp_annex

    rows_ex.append({
        'country': df['CTY_NAME'].iloc[0] if 'CTY_NAME' in df.columns else row.get('CTY_NAME', cty),
        'total_imports': total,
        'section_232': imp_232,
        'annex_exempt': imp_annex,
        'usmca_exempt': imp_usmca,
        'subject_to_surcharge': imp_subject,
    })

exemption_df = pd.DataFrame(rows_ex)

# ── Totals ──────────────────────────────────────────────────────────────
totals = exemption_df[['total_imports', 'section_232', 'annex_exempt',
                        'usmca_exempt', 'subject_to_surcharge']].sum()

print("Section 122 Surcharge — Import Coverage (2024 import weights)")
print("=" * 65)
print(f"Total imports (top 50 + EU):  ${totals['total_imports']/1e9:>10.1f}B")
print(f"  Section 232 carved out:     ${totals['section_232']/1e9:>10.1f}B  "
      f"({100*totals['section_232']/totals['total_imports']:>5.1f}%)")
print(f"  Annex I product exemptions: ${totals['annex_exempt']/1e9:>10.1f}B  "
      f"({100*totals['annex_exempt']/totals['total_imports']:>5.1f}%)")
print(f"  USMCA exempt (CA/MX):       ${totals['usmca_exempt']/1e9:>10.1f}B  "
      f"({100*totals['usmca_exempt']/totals['total_imports']:>5.1f}%)")
print(f"  Subject to 10% surcharge:   ${totals['subject_to_surcharge']/1e9:>10.1f}B  "
      f"({100*totals['subject_to_surcharge']/totals['total_imports']:>5.1f}%)")
print()

# ── Country breakdown ───────────────────────────────────────────────────
exemption_df['pct_subject'] = 100 * exemption_df['subject_to_surcharge'] / exemption_df['total_imports']
exemption_df['pct_exempt_annex'] = 100 * exemption_df['annex_exempt'] / exemption_df['total_imports']
exemption_df = exemption_df.sort_values('subject_to_surcharge', ascending=False)

print("Top 15 countries by imports subject to surcharge:")
print(exemption_df[['country', 'total_imports', 'subject_to_surcharge',
                     'pct_subject', 'annex_exempt', 'pct_exempt_annex']]
      .head(15).to_string(index=False, float_format='${:,.0f}'.format))


### Top countries: tariff change with Section 122

In [ ]:
# ── Top-X country change: 02-07 to 02-24 ─────────────────────────────────
TOP_X = 20  # set X here
TOP_BY = "imports_2025"  # options: "imports_2025" or "change"

START_DATE = pd.to_datetime("2026-02-07")
END_DATE = pd.to_datetime("2026-02-24")

panel = tariffs_by_date.reset_index().copy()

start_vals = (
    panel.loc[panel["date"] == START_DATE, ["country_name", "effective tariff"]]
    .rename(columns={"effective tariff": "start_tariff"})
)
end_vals = (
    panel.loc[panel["date"] == END_DATE, ["country_name", "effective tariff"]]
    .rename(columns={"effective tariff": "end_tariff"})
)

exclude_countries = {"CANADA", "MEXICO"}

rows_imports = []
for _, row in country_list.iterrows():
    df_2025 = load_imports(row["CTY_CODE"], year=2025)
    rows_imports.append({
        "country_name": df_2025["CTY_NAME"].iloc[0],
        "imports_2025": df_2025["imports"].sum(),
    })
imports_2025 = pd.DataFrame(rows_imports)

base = (
    start_vals.merge(end_vals, on="country_name", how="inner")
    .merge(imports_2025, on="country_name", how="left")
    .loc[lambda d: ~d["country_name"].str.upper().isin(exclude_countries)]
    .assign(change=lambda d: d["end_tariff"] - d["start_tariff"])
    .dropna(subset=["imports_2025"])
)

if TOP_BY == "imports_2025":
    chg = base.sort_values("imports_2025", ascending=False).head(TOP_X).sort_values("change", ascending=False)
    title_suffix = "by 2025 Import Volume"
else:
    chg = base.sort_values("change", key=lambda s: s.abs(), ascending=False).head(TOP_X).sort_values("change", ascending=False)
    title_suffix = "Top by absolute tariff change"

fig, ax = plt.subplots(figsize=(12, max(7, 0.35 * TOP_X)))
colors = np.where(chg["change"] >= 0, "tab:red", "tab:blue")
ax.barh(chg["country_name"], chg["change"], color=colors, alpha=0.9)
ax.axvline(0, color="black", linewidth=1.5, linestyle="--")

ax.invert_xaxis()  # tariff up to the left; tariff down to the right
ax.set_xlabel("Change in effective tariff rate (percentage points)", fontsize=11)
ax.set_ylabel("Country", fontsize=11)
ax.set_title(
    f"Top {TOP_X} Countries {title_suffix}: Change in Tarffs with S122 @ 10%",
    fontsize=13,
    loc="left",
)
ax.grid(axis="x", linestyle="--", alpha=0.35)
ax.spines["right"].set_visible(False)
ax.spines["top"].set_visible(False)

plt.tight_layout()
plt.show()

### Pre/Post SCOTUS + Section 122 summary

In [ ]:
# Pre/Post SCOTUS + Section 122 summary (trade-weighted)
scotus_date = pd.to_datetime("2026-02-20")
s122_date = pd.to_datetime("2026-02-24")

# Use the last modeled policy date before SCOTUS
pre_date = tariffs_by_date.index[tariffs_by_date.index < scotus_date].max()


def _weighted_stats_for_date(target_date):
    panel = tariffs_by_date.loc[target_date].copy()
    if isinstance(panel, pd.Series):
        panel = panel.to_frame().T

    # Drop synthetic total row if present
    panel = panel[panel["country_name"] != "TOTAL FOR ALL COUNTRIES"].copy()

    rates = panel["effective tariff"] + 2.3  # include MFN baseline for level comparability
    weights = panel["total imports"]

    weighted_mean = np.average(rates, weights=weights)
    weighted_var = np.average((rates - weighted_mean) ** 2, weights=weights)

    return weighted_mean, weighted_var

pre_mean, pre_var = _weighted_stats_for_date(pre_date)
post_mean, post_var = _weighted_stats_for_date(scotus_date)
s122_mean, s122_var = _weighted_stats_for_date(s122_date)

scotus_summary = pd.DataFrame([
    {
        "period": "Pre-SCOTUS",
        "date": pre_date.date(),
        "trade_weighted_avg_tariff_pct": pre_mean,
        "trade_weighted_country_variance": pre_var,
    },
    {
        "period": "Post-SCOTUS",
        "date": scotus_date.date(),
        "trade_weighted_avg_tariff_pct": post_mean,
        "trade_weighted_country_variance": post_var,
    },
    {
        "period": "S122",
        "date": s122_date.date(),
        "trade_weighted_avg_tariff_pct": s122_mean,
        "trade_weighted_country_variance": s122_var,
    },
]).set_index("period")

scotus_summary.round(2)